In [ ]:
#%pip install torch
#%pip install EduCDM
#%pip install EduData
#%pip install matplotlib

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from EduCDM import NCDM

from sklearn.model_selection import train_test_split
from collections import OrderedDict
from itertools import product

from cognitive_simulator import *

# NCDM Ability and Item Parameter Recovery

This notebook evaluates how accurately an NCDM (Neural Cognitive Diagnosis Model) can recover latent student abilities and item-skill dependency parameters from simulated data.

## Workflow

### 1. Data Generation 
### Example of dataset structure

Each row represents a user–exercise interaction with latent abilities and exercise-specific parameters.

---

##### 1. Core interaction data

| person_id | exercise_id | result |
|----------|-------------|--------|
| 1        | 1           | 0.0000 |
| 1        | 2           | 0.4334 |
| 1        | 3           | 0.0522 |
| 1        | 4           | 0.0023 |
| 1        | 5           | 0.1243 |

---

##### 2. User latent abilities

| ability_1 | ability_2 | ability_3 | ability_4 | ability_5 |
|----------|-----------|-----------|-----------|-----------|
| 1.0466   | 0.8480    | 0.6818    | 1.0745    | 0.8860    |

---

##### 3. Exercise skill parameters (example for one exercise)
Each exercise is associated with a subset of skills. Not all skills are required for every exercise.

| exercise_skill_1 | exercise_weight_1 | exercise_skill_2 | exercise_weight_2 |
|-----------------|-------------------|------------------|-------------------|
| 1.1270          | 0.1954            | 1.2186           | 0.4039            |
| 1.0334          | 0.9864            | 1.1328           | 0.2099            |

---
#### 4. Sparsity assumption (important)

Each exercise does **not necessarily use all available skills**.

- If a skill is not used in an exercise:
  - `exercise_skill_i = 0`
  - `exercise_weight_i = 0`

This enforces a **sparse Q-matrix structure**, where each exercise only activates a subset of the full skill space.

---

##### Notes

- Abilities are per-user latent variables.
- Each exercise activates a subset of skills.
- Skill importance is encoded via `(exercise_skill_i, exercise_weight_i)` pairs.

### 2. Data Loading

The simulation dataset is loaded from a CSV file into a pandas DataFrame.

Each row represents a student-item interaction and contains:

- Student identifier (`person_id`)
- Item identifier (`exercise_id`)
- Response outcome (`result`)
- Ground-truth student abilities (`ability_i`)
- Ground-truth item-skill dependency values (`exercise_skill_i`)
- Item weights (`exercise_weight_i`)

---

### 3. Dataset Transformation

The raw DataFrame is transformed into three structures:

#### `Q_matrix`

Maps items to the skills required for solving them.

| item_id | knowledge_code |
|----------|----------------|
| 1 | [1, 2] |
| 2 | [2] |
| 3 | [1, 3] |

#### `thetas`

Stores the true latent abilities of students.

| person_id | skills |
|------------|---------|
| 1 | [0.83, 1.12] |
| 2 | [1.01, 0.94] |

#### `results`

Stores student responses.

| user_id | item_id | score |
|----------|----------|--------|
| 1 | 1 | 1 |
| 1 | 2 | 0 |

---

### 4. Knowledge Structure Construction

Two helper structures are created:

#### `item2knowledge`

Maps each item to its associated skills.

```python
{
    1: [1, 2],
    2: [2],
    3: [1]
}
```

#### `knowledge_set`

Contains all unique skill identifiers.

```python
{1, 2, 3}
```

The number of skills is obtained as:

```python
n_abilities = len(knowledge_set)
```

---

### 5. PyTorch Dataset Preparation

The interaction data is converted into PyTorch datasets.

For each item, a multi-hot skill vector is generated:

```python
[1, 0, 1, 0]
```

where:

- `1` indicates the skill is required
- `0` indicates the skill is not required

The transformed data is wrapped into:

- `TensorDataset`
- `DataLoader`

for efficient mini-batch training.

The following loaders are created:

- `train_set`
- `valid_set`
- `test_set`

---

### 6. Model Training

The NCDM model is trained using:

```python
model.train(
    train_set,
    valid_set,
    epoch=10,
    device="cpu"
)
```

After training, the model is saved as:

```python
ncdm.snapshot
```

---

### 7. Item Parameter Recovery

Item difficulty embeddings are extracted from the model.
The embeddings are:

1. Passed through a sigmoid transformation.
2. Rescaled to the original latent ability distribution.

The rescaling formula is:

$$
\beta_{scaled}
=
1 + 0.15 \cdot
\frac{\beta_{sigmoid}-0.5}{0.2887}
$$

The resulting values represent recovered student abilities.


1. Sigmoid transformation is applied.
2. Values are rescaled back to the original parameter distribution.

Recovered item-skill dependency values are then compared to the true simulation parameters.

---
### 8. Ability Recovery (optional)

Student embeddings are extracted from the trained model.

The embeddings are:

1. Passed through a sigmoid transformation.
2. Rescaled to the original latent ability distribution.

The rescaling formula is:

$$
\theta_{scaled}
=
1 + 0.15 \cdot
\frac{\theta_{sigmoid}-0.5}{0.2887}
$$

The resulting values represent recovered student abilities.

---

### 9. Evaluation

The notebook evaluates recovery quality at three levels.

#### Item-Level Evaluation

Using:

```python
compare_betas(...)
```

Metrics:

- Item Mean Absolute Error (MAE)
- Item Mean Squared Error (MSE)

---



#### Student-Level Evaluation

Using:

```python
compare_student_abilities(...)
```

Metrics:

- Student Mean Absolute Error (MAE)
- Student Mean Squared Error (MSE)

---

#### Skill-Level Evaluation

Using:

```python
compare_thetas_uniquely(...)
```

Metrics:

- Average absolute error per skill
- Average squared error per skill
- Maximum absolute error per skill
- Maximum squared error per skill

---

## Output

The notebook produces:

- Recovered student abilities
- Recovered item-skill dependency values
- Item-level error metrics
- Student-level error metrics
- Skill-level error metrics

These results quantify how accurately the NCDM model reconstructs the latent parameters used to generate the simulated dataset.

In [ ]:
def transform_data(data, min_val):
    
    ability_cols = [c for c in data.columns if c.startswith("ability_")]
    ex_skill_cols = [c for c in data.columns if c.startswith("exercise_skill_")]
    ex_weight_cols = [c for c in data.columns if c.startswith("exercise_weight_")]

    n_abilities = len(ex_skill_cols)

    n_exercises = data["exercise_id"].max()

    Q_matrix = pd.DataFrame(columns=["item_id", "knowledge_code"])
    thetas = pd.DataFrame(columns=["person_id", "skills"])
    results = pd.DataFrame(columns=["user_id", "item_id", "score"])

    for i in range(len(data)):

        skills = data.loc[i, ex_skill_cols].values

        ex_skills = np.where(skills > min_val)[0] + 1

        if len(ex_skills) == 0:
            ex_skills = np.where(skills == skills.max())[0] + 1

        if i % n_exercises == 0:   #every student performs every exercise, so the student abilities are observed once per all exercises
            theta_i = data.loc[i, ability_cols].values
            person_id = int(data.loc[i, "person_id"])

            thetas.loc[person_id] = [person_id, theta_i]

        result = int(data.loc[i, "result"] > 0.5)

        if i < n_exercises:
            Q_matrix.loc[i] = [i + 1, ex_skills]

        results.loc[i] = [
            data.loc[i, "person_id"],
            data.loc[i, "exercise_id"],
            result
        ]

    results = results.astype(int)

    return results, Q_matrix, thetas

In [ ]:
def def_Q_matrix(Q_matrix):
    item2knowledge = {}
    knowledge_set = set()
    for i, s in Q_matrix.iterrows():
        item_id, knowledge_codes = s["item_id"], list(set(s["knowledge_code"]))
        item2knowledge[item_id] = knowledge_codes
        knowledge_set.update(knowledge_codes)

    return item2knowledge, knowledge_set

In [ ]:
def num_user_item_knowledge(train_data, val_data, test_data, knowledge_set):
    user_n = np.max(train_data["user_id"])
    item_n = np.max(
        [
            np.max(train_data["item_id"]),
            np.max(val_data["item_id"]),
            np.max(test_data["item_id"]),
        ]
    )
    n_abilities = np.max(list(knowledge_set))

    return user_n, item_n, n_abilities

In [ ]:
def split_train_test(results):

    train_data, test_data = train_test_split(results, test_size=0.2)
    train_data, val_data = train_test_split(train_data, test_size=0.25)
    train_data.index = range(0, len(train_data))
    val_data.index = range(0, len(val_data))
    test_data.index = range(0, len(test_data))

    return train_data, val_data, test_data

In [ ]:
def transform(user, item, item2knowledge, score, n_abilities, batch_size=32):
    knowledge_emb = torch.zeros((len(item), n_abilities))
    for idx in range(len(item)):
        knowledge_emb[idx][np.array(item2knowledge[item[idx]]) - 1] = 1.0

    data_set = TensorDataset(
        torch.tensor(user, dtype=torch.int64) - 1,  # (1, user_n) to (0, user_n-1)
        torch.tensor(item, dtype=torch.int64) - 1,  # (1, item_n) to (0, item_n-1)
        knowledge_emb,
        torch.tensor(score, dtype=torch.float32),
    )
    return DataLoader(data_set, batch_size=batch_size, shuffle=True)

In [ ]:
def transform_to_set(train_data, val_data, test_data, item2knowledge, n_abilities):
    train_set, valid_set, test_set = [
        transform(
            data["user_id"],
            data["item_id"],
            item2knowledge,
            data["score"],
            n_abilities=n_abilities,
            batch_size=32,
        )
        for data in [train_data, val_data, test_data]
    ]
    return train_set, valid_set, test_set

In [ ]:
def train_model(model, train_set, valid_set, item_n, user_n):
    model = model
    model.train(train_set, valid_set, epoch=10, device="cpu")
    model.save("ncdm.snapshot")

In [ ]:
def scale_thetas(model):
    student_proficiency_vectors = (
        model.ncdm_net.student_emb.weight.detach().cpu().numpy()
    )

    real_thetas = 1 / (1 + np.exp(-student_proficiency_vectors))
    mean_real, std_real = 0.5, 0.2887
    mean_orig, std_orig = 1, 0.15

    scaled_thetas = mean_orig + std_orig * (real_thetas - mean_real) / std_real

    return scaled_thetas

In [ ]:
def compare_betas(model, data, Q_matrix, n_abilities):

    skill_cols = [c for c in data.columns if c.startswith("exercise_skill_")]
    weight_cols = [c for c in data.columns if c.startswith("exercise_weight_")]

    # model difficulty embeddings
    exercise_proficiency_vectors = (
        model.ncdm_net.k_difficulty.weight.detach().cpu().numpy()
    )

    # sigmoid
    real_exercise_values = 1 / (1 + np.exp(-exercise_proficiency_vectors))
    # scale
    mean_real, std_real = 0.5, 0.2887
    mean_orig, std_orig = 1, 0.15

    scaled_k_diff_values = (
        mean_orig
        + std_orig * (real_exercise_values - mean_real) / std_real
    )

    exercise_skills_compare = []

    for i in range(len(Q_matrix)):

        # Q-matrix skill indices (0-based)
        idxs = Q_matrix.loc[i, "knowledge_code"] - 1

        # original weights from dataset
        original_skills = data.loc[i, skill_cols].values

        # model values for those skills
        model_values = scaled_k_diff_values[i][idxs]

        # matching original values
        original_values = original_skills[idxs]

        exercise_skills_compare.append((model_values, original_values))

    errors = [
        np.array(orig) - np.array(model)
        for model, orig in exercise_skills_compare
    ]

    return errors

In [ ]:
def compare_thetas_uniquely(model, real_thetas, n_abilities, scaled_thetas):

    real_theta = [elt for elt in real_thetas["skills"]]
    errors = {}
    for i in range(1, n_abilities + 1):
        errors[i] = []
    for real_vals, model in zip(real_theta, scaled_thetas):
        for j in range(n_abilities):
            e = model[j] - real_vals[j]
            errors[j + 1].append(e)

    return errors

In [ ]:
#Additional: student skills: comparison of original and model-predicted values
def compare_student_abilities(model, data, n_abilities, scaled_thetas):
    comparison = []
    for i in range(len(scaled_thetas)):
        comparison.append(data.loc[i][3 : (3 + n_abilities)] - scaled_thetas[i])
    return np.array(comparison, dtype=object)

In [ ]:
def summarize_errors(errors_dict, mode="abs_mean"):
    """
    errors_dict: {ability_id: np.array([...])}
    mode: abs_mean | sq_mean | abs_max | sq_max
    """
    out = OrderedDict()

    for aid, arr in errors_dict.items():
        arr = np.array(arr)

        if mode == "abs_mean":
            out[f"ability_{aid}_abs_mean"] = np.mean(np.abs(arr))
        elif mode == "sq_mean":
            out[f"ability_{aid}_sq_mean"] = np.mean(arr**2)
        elif mode == "abs_max":
            out[f"ability_{aid}_abs_max"] = np.max(np.abs(arr))
        elif mode == "sq_max":
            out[f"ability_{aid}_sq_max"] = np.max(arr**2)

    return out

In [ ]:
def run_test(n_students, n_exercises, n_abilities, filename, results_filename):

    # --- load results table ---
    df = pd.read_csv(results_filename)

    # --- generate data ---
    data_synthesis(
        n_students,
        n_exercises,
        n_abilities,
        binary=False,
        csv_name=filename
    )

    data = pd.read_csv(filename)

    # --- preprocessing ---
    results, Q_matrix, thetas = transform_data(data, 0.3)
    item2knowledge, knowledge_set = def_Q_matrix(Q_matrix)

    train_data, val_data, test_data = split_train_test(results)

    user_n, item_n, n_abilities = num_user_item_knowledge(
        train_data, val_data, test_data, knowledge_set
    )

    train, valid, test = transform_to_set(
        train_data, val_data, test_data,
        item2knowledge, n_abilities
    )

    # --- model ---
    model = NCDM(n_abilities, item_n, user_n)

    train_model(model, train, valid, item_n, user_n)

    auc, acc = model.eval(test)

    # --- item-skill recovery ---
    item_errors = compare_betas(model, data, Q_matrix, n_abilities)

    item_mean_abs_error = np.mean([
        np.mean(np.abs(e)) for e in item_errors
    ])

    item_s_e = np.mean([
        np.mean(e ** 2) for e in item_errors
    ])

    # --- final row ---
    row = {
        "nr_of_people": n_students,
        "nr_of_exercises": n_exercises,
        "nr_of_abilities": n_abilities,
        "accuracy": acc,
        "auc": auc,
        "item_mean_abs_errors": item_mean_abs_error,
        "item_avg_s_e": item_s_e
    }

    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)

    df.to_csv(results_filename, index=False)

In [ ]:
ABILITIES_RANGE = range(5, 11)
STUDENTS_RANGE = range(500, 1600, 100)
EXERCISES_RANGE = range(50, 110, 10)

In [ ]:
results_filename = "df_students_items.csv"
df = pd.DataFrame(columns=[])
df.to_csv(results_filename)

for n_abilities, n_students, n_exercises in product(
    ABILITIES_RANGE,
    STUDENTS_RANGE,
    EXERCISES_RANGE
):
    fname = f"test_results_{n_students}p_{n_abilities}a_{n_exercises}i.csv"

    run_test(
        n_students=n_students,
        n_exercises=n_exercises,
        n_abilities=n_abilities,
        filename=fname,
        results_filename=results_filename
    )